# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [1]:
import sys; print(sys.executable)


/Users/minamahdian/deploying-ai/.venv/bin/python


In [2]:
import sys
print(sys.executable)  # should be /Users/minamahdian/deploying-ai/.venv/bin/python

from langchain_community.document_loaders import PyPDFLoader, OnlinePDFLoader
print("✅ import OK")


/Users/minamahdian/deploying-ai/.venv/bin/python


/Users/minamahdian/deploying-ai/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ import OK


In [4]:
from langchain_community.document_loaders import OnlinePDFLoader

url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = OnlinePDFLoader(url)

# Fetch + extract text
docs = loader.load()

print(f"Loaded {len(docs)} pages.")
print(docs[0].page_content[:400])






A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/minamahdian/deploying-ai/.venv/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Us

Loaded 1 pages.
www.hbr.org

B

EST

OF HBR 1999

Success in the knowledge economy comes to those who know themselves—their strengths, their values, and how they best perform.

Managing Oneself

by Peter F. Drucker



Included with this full-text

Harvard Business Review

article:

1

Article Summary

The Idea in Brief—the core idea The Idea in Practice—putting the idea to work

2

Managing Oneself

12

Further R


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
from typing import Optional
from pydantic import BaseModel, Field
from openai import OpenAI

# -----------------------------
# 0) Pydantic schema (structured output)
# -----------------------------
class ArticleCard(BaseModel):
    Author: str = Field(..., description="Name(s) of the article's author(s)")
    Title: str = Field(..., description="Title of the article")
    Relevance: str = Field(
        ...,
        description="Why this article matters to AI professionals (<= 1 paragraph)."
    )
    Summary: str = Field(
        ...,
        description="Concise summary (<= 1000 tokens) in the specified tone."
    )
    Tone: str = Field(
        ...,
        description="The distinct tone used for the summary (e.g., 'Victorian English', 'Legalese', 'Formal Academic Writing')."
    )
    # Will fill these AFTER the model response using server-provided usage stats:
    InputTokens: Optional[int] = None
    OutputTokens: Optional[int] = None

# -----------------------------
# 1) Prompts kept separate
# -----------------------------
DEV_INSTRUCTIONS = """\
You are an assistant that returns ONLY valid JSON conforming to the provided schema.
- Do not include extra keys.
- Keep 'Relevance' to one paragraph.
- Keep 'Summary' under ~1000 tokens.
- Use a clearly identifiable tone (e.g., Victorian English, Legalese, Bureaucratese, AAVE, etc.).
- If context is incomplete, do your best with what is provided; do not fabricate citations.
"""

# You’ll pass context dynamically via .format(context=...)
USER_TASK_TEMPLATE = """\
You are given article context (title/author/content snippets or notes):

[CONTEXT START]
{context}
[CONTEXT END]

Task:
1) Produce a structured JSON object with fields: Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens.
2) Choose a distinct, recognizable Tone for the Summary (your choice, but be consistent).
3) Do NOT exceed one paragraph for Relevance.
4) The Summary should be concise but comprehensive (<=1000 tokens).
5) Do not include any commentary outside of the JSON object.
"""

# -----------------------------
# 2) Insert your dynamic context here
# -----------------------------
# Replace this with the article’s text/notes you have (or a brief extract).
context_str = """
Title: Managing Oneself
Author: Peter F. Drucker
Notes: Classic HBR piece on self-awareness, personal strengths, preferred learning styles,
and where to contribute most effectively for long-term impact. Often cited in leadership
and career growth, relevant to tech/AI professionals navigating evolving roles.
"""

# -----------------------------
# 3) Call the API with structured output
# -----------------------------
client = OpenAI()

# Use a model NOT in the GPT-5 family (example: gpt-4o-mini)
model_name = "gpt-4o-mini"

# The Responses API supports structured outputs with Pydantic via `response_format=ArticleCard`
# If your SDK version doesn't expose `responses.parse`, fall back to JSON schema mode (shown below).
try:
    # Preferred (if your SDK supports it):
    response = client.responses.parse(
        model=model_name,
        # Keep instructions (system) and user prompt separate:
        input=[
            {"role": "system", "content": DEV_INSTRUCTIONS},
            {"role": "user", "content": USER_TASK_TEMPLATE.format(context=context_str)},
        ],
        response_format=ArticleCard,  # Pydantic model
    )
    parsed: ArticleCard = response.output_parsed

except Exception:
    # Fallback: JSON schema route (older SDKs)
    from json import dumps
    json_schema = {
        "name": "ArticleCard",
        "schema": {
            "type": "object",
            "properties": {
                "Author": {"type": "string"},
                "Title": {"type": "string"},
                "Relevance": {"type": "string"},
                "Summary": {"type": "string"},
                "Tone": {"type": "string"},
                "InputTokens": {"type": ["integer", "null"]},
                "OutputTokens": {"type": ["integer", "null"]},
            },
            "required": ["Author", "Title", "Relevance", "Summary", "Tone"],
            "additionalProperties": False,
        },
        "strict": True,
    }

    response = client.responses.create(
        model=model_name,
        input=[
            {"role": "system", "content": DEV_INSTRUCTIONS},
            {"role": "user", "content": USER_TASK_TEMPLATE.format(context=context_str)},
        ],
        response_format={"type": "json_schema", "json_schema": json_schema},
    )

    # Parse JSON manually into the Pydantic model
    import json
    raw_json = response.output[0].content[0].text  # SDK shape may vary; adjust if needed
    parsed = ArticleCard(**json.loads(raw_json))

# -----------------------------
# 4) Fill in token usage from server response, then print as JSON
# -----------------------------
usage = getattr(response, "usage", None)
if usage:
    parsed.InputTokens = getattr(usage, "input_tokens", None)
    parsed.OutputTokens = getattr(usage, "output_tokens", None)

# Final result as a Pydantic object (with token counts from the response):
print(parsed.model_dump_json(indent=2, ensure_ascii=False))


ModuleNotFoundError: No module named 'openai'

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
